# Evaluation Notebook (Load-Only, CBF IVF + CF ALS + Ranx)

Notebook nay danh gia offline theo nguyen tac load-only:
- KHONG tao moi FAISS index trong notebook evaluate.
- Tai va danh gia 2 model: `cbf_ivf` va `cf_als`.
- Dung Ranx cho Recall@K, NDCG@K, MRR@K.
- Bao cao them Coverage@K va theo segment user.

Dataset train/test: `data/processed/eval`
Artifacts index: `data/processed/test`

In [1]:
# Cell 2 - Optional Colab mount
IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    print('Drive mounted.')
else:
    print('Running outside Colab. Skip Drive mount.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [2]:
# Cell 3 - Optional dependency install
INSTALL_DEPS = False
if INSTALL_DEPS:
    if IN_COLAB:
        %pip install -q numpy pandas pyarrow polars tqdm ranx faiss-gpu-cu12
    else:
        %pip install -q numpy pandas pyarrow polars tqdm ranx faiss-cpu

In [3]:
from __future__ import annotations

import gc
import json
import logging
import sys
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import polars as pl
from ranx import Qrels, Run, evaluate
from tqdm.auto import tqdm

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    force=True,
 )
logger = logging.getLogger('evaluation_ivf_cf_ranx')

def infer_base_dir(in_colab: bool) -> Path:
    if in_colab:
        return Path('/content/drive/My Drive/Thesis/book_recsys')
    cwd = Path.cwd().resolve()
    return cwd if (cwd / 'app').exists() else (cwd / 'book_recsys')

@dataclass
class EvalConfig:
    base_dir: str = ''
    train_path: str = 'data/processed/eval/interactions_train.csv'
    test_path: str = 'data/processed/eval/interactions_test.csv'

    # Shared artifacts
    mapping_path: str = 'data/processed/test/cb_sbert_v1_book_index.csv'
    user_mapping_path: str = 'data/processed/test/cb_sbert_v1_user_index.csv'
    user_history_path: str = 'data/processed/test/cb_sbert_v1_user_histories.parquet'

    # CBF artifacts
    cbf_index_path: str = 'data/processed/test/cb_sbert_v1_IVFFlat.index'
    cbf_item_vectors_path: str = 'data/processed/test/cb_sbert_v1_matrix.npy'
    cbf_user_profiles_path: str = 'data/processed/test/cb_sbert_v1_user_profiles_mean.npy'

    # CF artifacts (load-only: index must already exist)
    cf_index_path: str = 'data/processed/test/cf_als_v1_FlatIP.index'
    cf_user_profiles_path: str = 'data/processed/test/cf_als_v1_user_profiles.npy'

    vector_dim: int = 384
    k: int = 10
    topn: int = 100
    chunk_size: int = 4000

    cold_start_max_interactions: int = 4
    heavy_min_interactions: int = 20

    eval_device: str = 'gpu'
    use_gpu: bool = True
    model_aliases: list[str] = field(default_factory=lambda: ['cbf_ivf', 'cf_als'])
    enable_ranx_compare: bool = True

    artifact_root: str = 'artifacts/evaluation'

    def resolve(self, rel_or_abs: str) -> Path:
        path = Path(rel_or_abs)
        return path if path.is_absolute() else Path(self.base_dir) / path

cfg = EvalConfig()
if not cfg.base_dir:
    cfg.base_dir = str(infer_base_dir(IN_COLAB))

project_root = Path(cfg.base_dir)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.recommenders import MeanProfileManager, RecommendRequest, RecommenderFactory
from app.recommenders.item_profile import ItemProfile

print('Project root:', project_root)

2026-04-15 22:25:48,317 | INFO | Loading faiss with AVX512 support.
2026-04-15 22:25:48,319 | INFO | Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-04-15 22:25:48,320 | INFO | Loading faiss with AVX2 support.
2026-04-15 22:25:48,321 | INFO | Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-04-15 22:25:48,323 | INFO | Loading faiss.
2026-04-15 22:25:48,403 | INFO | Successfully loaded faiss.


Project root: /content/drive/My Drive/Thesis/book_recsys


In [4]:
# Cell 5 - Load train/test and build qrels + user segments
def _normalize_interactions(df: pl.DataFrame) -> pl.DataFrame:
    item_col = next((c for c in ('item_id', 'book_id', 'id') if c in df.columns), 'item_id')
    if item_col != 'item_id':
        df = df.rename({item_col: 'item_id'})
    return df.select([
        pl.col('user_id').cast(pl.Utf8),
        pl.col('item_id').cast(pl.Utf8),
    ])

def load_eval_maps(config: EvalConfig):
    train_df = _normalize_interactions(pl.read_csv(config.resolve(config.train_path)))
    test_df = _normalize_interactions(pl.read_csv(config.resolve(config.test_path)))

    qrels_grouped = test_df.group_by('user_id').agg(pl.col('item_id'))
    qrels_dict = {
        row['user_id']: {str(item_id): 1 for item_id in row['item_id']}
        for row in qrels_grouped.iter_rows(named=True)
    }

    train_grouped = train_df.group_by('user_id').agg([
        pl.col('item_id').alias('seen_items'),
        pl.len().alias('n_train_interactions'),
    ])

    seen_map: dict[str, set[str]] = {}
    segment_map: dict[str, str] = {}

    for row in train_grouped.iter_rows(named=True):
        user_id = row['user_id']
        n_interactions = int(row['n_train_interactions'])
        seen_map[user_id] = set(str(item_id) for item_id in row['seen_items'])

        if n_interactions <= config.cold_start_max_interactions:
            segment_map[user_id] = 'COLD_START'
        elif n_interactions >= config.heavy_min_interactions:
            segment_map[user_id] = 'HEAVY'
        else:
            segment_map[user_id] = 'LIGHT'

    eval_users = sorted(set(qrels_dict.keys()) & set(seen_map.keys()))
    filtered_qrels = {user_id: qrels_dict[user_id] for user_id in eval_users}

    segment_distribution_df = (
        pd.Series([segment_map[u] for u in eval_users], name='segment')
        .value_counts()
        .sort_index()
        .rename_axis('segment')
        .reset_index(name='n_users')
    )

    return filtered_qrels, seen_map, segment_map, eval_users, segment_distribution_df, train_df.height, test_df.height

qrels_dict, seen_map, segment_map, eval_users, segment_distribution_df, n_train_rows, n_test_rows = load_eval_maps(cfg)
qrels = Qrels(qrels_dict)

print(f'train rows: {n_train_rows:,}')
print(f'test rows: {n_test_rows:,}')
print(f'eval users: {len(eval_users):,}')
display(segment_distribution_df)

train rows: 15,498,584
test rows: 4,049,341
eval users: 274,613


,segment,n_users
0,COLD_START,22737
1,HEAVY,152409
2,LIGHT,99467


In [5]:
# Cell 6 - Helpers for load-only model evaluation
def _assert_exists(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f'Missing {label}: {path}')

def load_model_bundle(model_alias: str, config: EvalConfig):
    _assert_exists(config.resolve(config.mapping_path), 'item mapping')
    _assert_exists(config.resolve(config.user_mapping_path), 'user mapping')
    _assert_exists(config.resolve(config.user_history_path), 'user history')

    if model_alias == 'cbf_ivf':
        index_path = config.resolve(config.cbf_index_path)
        item_vectors_path = config.resolve(config.cbf_item_vectors_path)
        user_profiles_path = config.resolve(config.cbf_user_profiles_path)
        _assert_exists(index_path, 'CBF index')
        _assert_exists(item_vectors_path, 'CBF item vectors')
        _assert_exists(user_profiles_path, 'CBF user profiles')

        item_profile = ItemProfile(
            mapping_path=str(config.resolve(config.mapping_path)),
            vectors_path=str(item_vectors_path),
            vector_dim=config.vector_dim,
        )
        model_type = 'sbert'
        model_name = 'cbf_faiss_ivf'

    elif model_alias == 'cf_als':
        index_path = config.resolve(config.cf_index_path)
        user_profiles_path = config.resolve(config.cf_user_profiles_path)
        _assert_exists(index_path, 'CF index')
        _assert_exists(user_profiles_path, 'CF user profiles')

        item_profile = ItemProfile(
            mapping_path=str(config.resolve(config.mapping_path)),
            vectors_path=None,
            vector_dim=config.vector_dim,
        )
        model_type = 'cf'
        model_name = 'cf_als_faiss'

    else:
        raise ValueError(f'Unsupported model_alias: {model_alias}')

    item_profile.load()

    profile_manager = MeanProfileManager(
        user_vectors_path=str(user_profiles_path),
        user_mapping_path=str(config.resolve(config.user_mapping_path)),
        user_history_path=str(config.resolve(config.user_history_path)),
        vector_dim=config.vector_dim,
        implicit_weight=0.2,
        use_rating_scale=False,
        use_gpu=config.use_gpu,
    )
    profile_manager.load()

    model = RecommenderFactory.create(
        model_type,
        index_path=str(index_path),
        item_profile=item_profile,
        user_profile_manager=profile_manager,
        model_name=model_name,
        gpu_device_id=0,
        fallback_to_cpu_on_gpu_error=True,
    )
    model.load()
    model.set_runtime(device=config.eval_device, use_gpu=config.use_gpu)

    return model, profile_manager

def predict_one_model(
    model_alias: str,
    model,
    profile_manager: MeanProfileManager,
    config: EvalConfig,
    users: list[str],
    seen_items_map: dict[str, set[str]],
    user_segments: dict[str, str],
):
    run_dict: dict[str, dict[str, float]] = {}
    rows: list[dict[str, object]] = []
    skipped_no_profile = 0

    for start in tqdm(range(0, len(users), config.chunk_size), desc=f'Predict {model_alias}'):
        batch_users = users[start:start + config.chunk_size]
        batch_requests = []

        for user_id in batch_users:
            user_vector = profile_manager.get_profile(user_id)
            if user_vector is None:
                skipped_no_profile += 1
                continue

            batch_requests.append(
                RecommendRequest(
                    user_id=user_id,
                    user_vector=user_vector,
                    seen_item_ids=seen_items_map.get(user_id, set()),
                )
            )

        if not batch_requests:
            continue

        # CFRecommender must use per-user path to preserve ALS dot-product magnitude.
        if model_alias == 'cf_als':
            outputs = {
                req.user_id: model.recommend(req, k=config.k, topn=config.topn)
                for req in batch_requests
            }
        else:
            outputs = model.recommend_batch(batch_requests, k=config.k, topn=config.topn)

        for user_id, recs in outputs.items():
            run_dict[user_id] = {rec.item_id: float(rec.score) for rec in recs}
            segment = user_segments.get(user_id, 'UNKNOWN')
            for rec in recs:
                rows.append({
                    'model_alias': model_alias,
                    'model_name': model.model_name,
                    'user_id': user_id,
                    'segment': segment,
                    'item_id': rec.item_id,
                    'score': float(rec.score),
                    'rank': int(rec.rank),
                })

    pred_df = pd.DataFrame(rows)
    if not pred_df.empty:
        pred_df = pred_df.sort_values(['model_alias', 'user_id', 'rank']).reset_index(drop=True)

    return run_dict, pred_df, skipped_no_profile

In [ ]:
# Cell 7 - Sequential evaluate (memory-safe for multi-model)
metrics = [f'recall@{cfg.k}', f'ndcg@{cfg.k}', f'mrr@{cfg.k}']

pred_frames: list[pd.DataFrame] = []
overall_rows: list[dict[str, object]] = []
segment_rows: list[dict[str, object]] = []

served_users_by_model: dict[str, int] = {}
skipped_users_by_model: dict[str, int] = {}
ranx_runs: dict[str, Run] = {}

for model_alias in cfg.model_aliases:
    logger.info('Evaluating model: %s', model_alias)
    model = None
    profile_manager = None
    run_dict = {}
    model_pred_df = pd.DataFrame()

    try:
        model, profile_manager = load_model_bundle(model_alias, cfg)
        run_dict, model_pred_df, skipped = predict_one_model(
            model_alias=model_alias,
            model=model,
            profile_manager=profile_manager,
            config=cfg,
            users=eval_users,
            seen_items_map=seen_map,
            user_segments=segment_map,
        )

        served_users_by_model[model_alias] = len(run_dict)
        skipped_users_by_model[model_alias] = skipped

        if model_pred_df.empty:
            logger.warning('No predictions for model=%s', model_alias)
            continue
        pred_frames.append(model_pred_df)

        eval_users_model = [u for u in run_dict.keys() if u in qrels_dict]
        if not eval_users_model:
            logger.warning('No overlap with qrels for model=%s', model_alias)
            continue

        model_qrels = Qrels({u: qrels_dict[u] for u in eval_users_model})
        model_run_dict = {u: run_dict[u] for u in eval_users_model}
        model_run = Run(model_run_dict, name=model.model_name)
        model_scores = evaluate(model_qrels, model_run, metrics=metrics, make_comparable=True)

        item_total = max(int(model.item_profile.total_items), 1)
        coverage = float(model_pred_df['item_id'].nunique() / item_total)

        overall_rows.append({
            'model_alias': model_alias,
            'model_name': model.model_name,
            'segment': 'OVERALL',
            'n_users': int(len(eval_users_model)),
            f'coverage@{cfg.k}': coverage,
            **{m: float(model_scores[m]) for m in metrics},
        })

        for segment_name in ('COLD_START', 'LIGHT', 'HEAVY'):
            segment_users = [u for u in eval_users_model if segment_map.get(u) == segment_name]
            if not segment_users:
                continue

            seg_qrels = Qrels({u: qrels_dict[u] for u in segment_users})
            seg_run_dict = {u: model_run_dict[u] for u in segment_users}
            seg_run = Run(seg_run_dict, name=f'{model.model_name}_{segment_name.lower()}')
            seg_scores = evaluate(seg_qrels, seg_run, metrics=metrics, make_comparable=True)

            seg_items = {iid for recs in seg_run_dict.values() for iid in recs.keys()}
            seg_coverage = float(len(seg_items) / item_total)

            segment_rows.append({
                'model_alias': model_alias,
                'model_name': model.model_name,
                'segment': segment_name,
                'n_users': int(len(segment_users)),
                f'coverage@{cfg.k}': seg_coverage,
                **{m: float(seg_scores[m]) for m in metrics},
            })

        if cfg.enable_ranx_compare:
            ranx_runs[model_alias] = model_run

    finally:
        del run_dict
        del model_pred_df
        if profile_manager is not None:
            del profile_manager
        if model is not None:
            del model
        gc.collect()

if not pred_frames:
    raise RuntimeError('No predictions were generated for any model.')

pred_df = pd.concat(pred_frames, ignore_index=True)
pred_df = pred_df.sort_values(['model_alias', 'user_id', 'rank']).reset_index(drop=True)
overall_metrics_df = pd.DataFrame(overall_rows).sort_values('model_alias').reset_index(drop=True)
segment_metrics_df = pd.DataFrame(segment_rows).sort_values(['model_alias', 'segment']).reset_index(drop=True)

display(overall_metrics_df)
display(segment_metrics_df)

2026-04-15 22:26:27,879 | INFO | Evaluating model: cbf_ivf
2026-04-15 22:26:29,406 | INFO | [ItemProfile] Load success 2360648 Items.
2026-04-15 22:26:29,698 | INFO | Loaded 465,037 User Mapping.
2026-04-15 22:26:29,702 | INFO | Loading User History Master (Parquet)...
2026-04-15 22:26:39,591 | INFO | Loaded history for 465,037 Users.
2026-04-15 22:26:39,592 | INFO | [MeanProfileManager] User storage system is ready.
2026-04-15 22:26:39,598 | INFO | [cbf_faiss_ivf] Loading FAISS index (MMAP)...
2026-04-15 22:26:39,827 | INFO | [cbf_faiss_ivf] Loaded 2360648 items.
2026-04-15 22:26:39,828 | INFO | Runtime set to device: gpu, use_gpu: True
2026-04-15 22:26:57,835 | INFO | [cbf_faiss_ivf] FAISS moved to GPU 0.


Predict cbf_ivf:   0%|          | 0/69 [00:00<?, ?it/s]

2026-04-15 22:29:45,302 | INFO | Evaluating model: cf_als
2026-04-15 22:29:46,609 | INFO | [ItemProfile] Load success 2360648 Items.
2026-04-15 22:29:46,910 | INFO | Loaded 465,037 User Mapping.
2026-04-15 22:29:46,914 | INFO | Loading User History Master (Parquet)...
2026-04-15 22:29:56,938 | INFO | Loaded history for 465,037 Users.
2026-04-15 22:29:56,939 | INFO | [MeanProfileManager] User storage system is ready.
2026-04-15 22:29:56,943 | INFO | [cf_als_faiss] Loading FAISS index (MMAP)...
2026-04-15 22:30:15,254 | INFO | [cf_als_faiss] Loaded 2360648 items.
2026-04-15 22:30:15,268 | INFO | Runtime set to device: gpu, use_gpu: True
2026-04-15 22:30:16,343 | INFO | [cf_als_faiss] FAISS moved to GPU 0.


Predict cf_als:   0%|          | 0/69 [00:00<?, ?it/s]

In [ ]:
# Cell 8 - Ranx direct model comparison (optional)
comparison_df = pd.DataFrame()

if cfg.enable_ranx_compare and len(ranx_runs) >= 2:
    try:
        from ranx import compare

        compare_result = compare(
            qrels=qrels,
            runs=list(ranx_runs.values()),
            metrics=metrics,
        )

        if isinstance(compare_result, pd.DataFrame):
            comparison_df = compare_result
        elif hasattr(compare_result, 'to_pandas'):
            comparison_df = compare_result.to_pandas()
        else:
            comparison_df = pd.DataFrame(compare_result)

        print('Ranx comparison completed.')
        display(comparison_df)
    except Exception as exc:
        logger.warning('Ranx compare failed: %s', exc)
else:
    print('Skip Ranx compare (need >=2 models and enable_ranx_compare=True).')

In [ ]:
# Cell 9 - Quick ranking view
if overall_metrics_df.empty:
    print('No overall metrics to display.')
else:
    ranking_df = overall_metrics_df.sort_values([f'ndcg@{cfg.k}', f'recall@{cfg.k}'], ascending=False)
    display(ranking_df[['model_alias', 'model_name', f'ndcg@{cfg.k}', f'recall@{cfg.k}', f'mrr@{cfg.k}', f'coverage@{cfg.k}', 'n_users']])

In [ ]:
# Cell 10 - Save artifacts
run_id = f"eval_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
run_dir = cfg.resolve(cfg.artifact_root) / run_id
run_dir.mkdir(parents=True, exist_ok=True)

pred_path = run_dir / 'predictions_topk.parquet'
overall_path = run_dir / 'metrics_overall.parquet'
segment_path = run_dir / 'metrics_by_segment.parquet'
compare_path = run_dir / 'metrics_compare.parquet'
summary_path = run_dir / 'metrics_summary.json'

pred_df.to_parquet(pred_path, index=False)
if not overall_metrics_df.empty:
    overall_metrics_df.to_parquet(overall_path, index=False)
if not segment_metrics_df.empty:
    segment_metrics_df.to_parquet(segment_path, index=False)
if not comparison_df.empty:
    comparison_df.to_parquet(compare_path, index=False)

summary = {
    'run_id': run_id,
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_aliases_requested': cfg.model_aliases,
    'model_aliases_evaluated': overall_metrics_df['model_alias'].tolist() if not overall_metrics_df.empty else [],
    'device_requested': cfg.eval_device,
    'use_gpu': cfg.use_gpu,
    'n_train_rows': int(n_train_rows),
    'n_test_rows': int(n_test_rows),
    'n_eval_users': int(len(eval_users)),
    'served_users_by_model': served_users_by_model,
    'skipped_users_by_model': skipped_users_by_model,
    'segment_distribution': segment_distribution_df.to_dict(orient='records'),
    'overall_metrics': overall_metrics_df.to_dict(orient='records') if not overall_metrics_df.empty else [],
    'has_ranx_compare': bool(not comparison_df.empty),
}

with summary_path.open('w', encoding='utf-8') as file_obj:
    json.dump(summary, file_obj, ensure_ascii=False, indent=2)

print('Saved:', pred_path)
if not overall_metrics_df.empty:
    print('Saved:', overall_path)
if not segment_metrics_df.empty:
    print('Saved:', segment_path)
if not comparison_df.empty:
    print('Saved:', compare_path)
print('Saved:', summary_path)